In [41]:
import os 
from glob import glob
import pandas as pd

In [42]:
# 특정 경로의 파일의 목록을 가져오는 기능
# os 라이브러리를 이용
os.listdir('./review_data')

['1-1.여성의류(208).json',
 '1-1.여성의류(197).json',
 '1-1.여성의류(196).json',
 '1-1.여성의류(209).json',
 '1-1.여성의류(198).json',
 '1-1.여성의류(205).json',
 '1-1.여성의류(207).json',
 '1-1.여성의류(203).json',
 '1-1.여성의류(201).json',
 '1-1.여성의류(206).json',
 '1-1.여성의류(199).json',
 '1-1.여성의류(204).json',
 '1-1.여성의류(200).json',
 '1-1.여성의류(202).json']

In [43]:
# glob 이용 
# 장점 : 파일의 경로와 파일의 이름을 하나의 리스트로 생성 
#       특정 확장자만 선택해서 리스트로 생성이 가능
json_list = glob("./review_data/*.json")

In [44]:
# json_list를 이용하여 하나의 데이터프레임으로 단순 행 결합

# 빈 데이터프레임을 생성
total_df = pd.DataFrame()

for file_path in json_list:
    # print(file_path)
    df = pd.read_json(file_path)
    # total_df, df를 단순 행결합을 하여 total_df에 대입 
    total_df = pd.concat( [total_df, df], axis=0 )
    # print(df)
    # break
total_df.reset_index(drop=True, inplace=True)

In [45]:
total_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1423 entries, 0 to 1422
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            1423 non-null   int64  
 1   RawText          1423 non-null   object 
 2   Source           1423 non-null   object 
 3   Domain           1423 non-null   object 
 4   MainCategory     1423 non-null   object 
 5   ProductName      1423 non-null   object 
 6   Syllable         1423 non-null   int64  
 7   Word             1423 non-null   int64  
 8   GeneralPolarity  1418 non-null   float64
 9   Aspects          1423 non-null   object 
dtypes: float64(1), int64(3), object(6)
memory usage: 111.3+ KB


In [46]:
pd.concat(
    [ pd.read_json(file_path) for file_path in json_list[:5] ]
).info()

<class 'pandas.core.frame.DataFrame'>
Index: 523 entries, 0 to 99
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            523 non-null    int64  
 1   RawText          523 non-null    object 
 2   Source           523 non-null    object 
 3   Domain           523 non-null    object 
 4   MainCategory     523 non-null    object 
 5   ProductName      523 non-null    object 
 6   Syllable         523 non-null    int64  
 7   Word             523 non-null    int64  
 8   GeneralPolarity  522 non-null    float64
 9   Aspects          523 non-null    object 
dtypes: float64(1), int64(3), object(6)
memory usage: 44.9+ KB


In [47]:
# Aspects 의 데이터를 하나로 합치고 새로운 데이터 프레임을 생성 
aspect_df = pd.DataFrame(sum(total_df['Aspects'], []))

In [48]:
aspect_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10962 entries, 0 to 10961
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Aspect             10962 non-null  object
 1   SentimentText      10962 non-null  object
 2   SentimentWord      10962 non-null  object
 3   SentimentPolarity  10962 non-null  object
dtypes: object(4)
memory usage: 342.7+ KB


In [49]:
# 데이터의 불균형 문제 확인 
aspect_df['SentimentPolarity'].value_counts()

SentimentPolarity
1     9664
-1    1005
0      293
Name: count, dtype: int64

In [50]:
aspect_df.isna().sum()

Aspect               0
SentimentText        0
SentimentWord        0
SentimentPolarity    0
dtype: int64

In [51]:
# 데이터셋에서 문자열의 좌우의 공백을 제거 
# 모든 컬럼이 Object 형이기 때문에 strip() 바로 사용 가능
aspect_df = aspect_df.map(lambda x : x.strip())

In [52]:
aspect_df.isin(['']).sum()

Aspect               0
SentimentText        0
SentimentWord        0
SentimentPolarity    0
dtype: int64

In [53]:
aspect_df['SentimentText'].value_counts()

SentimentText
가볍고                          38
따뜻하고                         25
저렴한 가격에                      15
가격도 저렴하고                     12
시원하고                         11
                             ..
양쪽 옆에 주머니가 있어 편리함까지 갖췄네요.     1
단추를 채우고 입으면 단정한 원피스로,         1
풀고 입으면 세련된 로브 스타일의 가디건으로      1
두가지 스타일로 연출이 가능한 굿템입니다.       1
올겨울 그럭저력 잘 입을듯합니다             1
Name: count, Length: 10467, dtype: int64

In [54]:
before_cnt = len(aspect_df)

aspect_df.drop_duplicates('SentimentText', inplace=True)

after_cnt = len(aspect_df)

print(f"제거가 된 행의 개수 {before_cnt - after_cnt}")

제거가 된 행의 개수 495


In [55]:
# 1, 0, -1 의 비율을 확인 
aspect_df['SentimentPolarity'].value_counts()

SentimentPolarity
1     9183
-1     994
0      290
Name: count, dtype: int64

In [56]:
# 인덱스를 초기화 
aspect_df.reset_index(drop=True, inplace=True)

In [57]:
# 토큰화 -> 백터화 
from konlpy.tag import Komoran
from sklearn.feature_extraction.text import TfidfVectorizer

komoran = Komoran()
allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'MAG', 'SL']

def komoran_tokenize(text):
    tokens = []
    for word, pos in komoran.pos(text):
        if (pos in allow_pos) & (len(word) >= 2)  :
            tokens.append(word)
    return tokens

vectorizer = TfidfVectorizer(
    tokenizer= komoran_tokenize, 
    ngram_range=(1, 2), 
    min_df = 3, 
    max_df=0.8, 
    max_features=30000
)

In [58]:
# 모델 생성 
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputClassifier

In [59]:
svc = LinearSVC(random_state=42, class_weight='balanced')

multi_model = MultiOutputClassifier(svc)

pipe = Pipeline(
    [
        ('vector', vectorizer), 
        ('model', multi_model)
    ]
)


In [60]:
# 계층화 폴드 
from sklearn.model_selection import KFold

skfold = KFold(n_splits=3, shuffle= True, 
                         random_state=42)

In [61]:
aspect_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10467 entries, 0 to 10466
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Aspect             10467 non-null  object
 1   SentimentText      10467 non-null  object
 2   SentimentWord      10467 non-null  object
 3   SentimentPolarity  10467 non-null  object
dtypes: object(4)
memory usage: 327.2+ KB


In [62]:
from sklearn.preprocessing import LabelEncoder

In [63]:
le = LabelEncoder()
aspect_df['Aspect'] = le.fit_transform(aspect_df['Aspect'])
aspect_df['SentimentPolarity'] = aspect_df[
    'SentimentPolarity'].astype('int')

In [64]:
# 독립 변수 , 종속 변수 생성
X = aspect_df['SentimentText'].values
Y = aspect_df[['Aspect', 'SentimentPolarity']].values

In [65]:
print(X.shape, Y.shape)

(10467,) (10467, 2)


In [66]:
print(type(Y[0][0]), type(Y[0][1]))

<class 'numpy.int64'> <class 'numpy.int64'>


In [67]:
from sklearn.model_selection import GridSearchCV

In [68]:
params = {
    'model__estimator__C' : [1.0, 2.0]
}
grid = GridSearchCV(
    estimator=pipe, 
    param_grid= params, 
    cv = skfold, 
    scoring="accuracy"
)

In [69]:
grid.fit(X, Y)

/Users/eunseo/Documents/data_boot/venv311/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/eunseo/Documents/data_boot/venv311/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:953: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/Users/eunseo/Documents/data_boot/venv311/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 942, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/eunseo/Documents/data_boot/venv311/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 308, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ^^^^^^^^^^^^^^^^^^^^

,estimator,Pipeline(step..._state=42)))])
,param_grid,"{'model__estimator__C': [1.0, 2.0]}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,input,'content'


In [70]:
grid.best_score_

np.float64(nan)

1. total_df에서 rawText 컬럼의 데이터들을 이용하여 grid의 best_estimator_에서 예측을 실행
2. 실행된 결과 값을 이용하여 데이터프레임(rawText, Aspect_pred, Pola_pred)으로 생성
3. rawText, Aspect_pred 값을 이용하여 그룹화 -> 그룹화 연산에는 평균

In [71]:
best_model = grid.best_estimator_

In [72]:
from konlpy.tag import Kkma

In [73]:
kkma = Kkma()
# raw_list 에는 리뷰 문단을 문장으로 나눈 리스트를 담기 위한 공간
raw_list = []
# raw_dict는 리뷰 문단마다 index를 키값으로 value는 리뷰 문단
raw_dict = {}
for i in range(len(total_df)):
    # print(kkma.sentences(total_df.loc[i, 'RawText']))
    # break
    raw_list.append(kkma.sentences(total_df.loc[i, 'RawText']))
    raw_dict[i] = total_df.loc[i, 'RawText']

In [74]:
sentence_df = pd.DataFrame()
for idx, raw in enumerate(raw_list):
    pred = best_model.predict(raw)
    temp_df = pd.DataFrame(pred, columns = ['Aspect_pred', 'Pola_pred'])
    temp_df['RawText'] = raw_dict[idx]
    sentence_df = pd.concat([sentence_df, temp_df])

In [75]:
sentence_df

,Aspect_pred,Pola_pred,RawText
0,1,1,안녕하세요. 벌써 7월이라니 시간 정말 빨리 가는거 같아요. 날씨는 또 왜이리 더운...
1,1,1,안녕하세요. 벌써 7월이라니 시간 정말 빨리 가는거 같아요. 날씨는 또 왜이리 더운...
2,1,1,안녕하세요. 벌써 7월이라니 시간 정말 빨리 가는거 같아요. 날씨는 또 왜이리 더운...
3,1,1,안녕하세요. 벌써 7월이라니 시간 정말 빨리 가는거 같아요. 날씨는 또 왜이리 더운...
4,13,1,안녕하세요. 벌써 7월이라니 시간 정말 빨리 가는거 같아요. 날씨는 또 왜이리 더운...
...,...,...,...
5,17,-1,예년보다 따뜻한 11월이더니 역시 겨울은 겨울입니다. 갑자기 기온이 뚝 떨어지더니 ...
6,8,-1,예년보다 따뜻한 11월이더니 역시 겨울은 겨울입니다. 갑자기 기온이 뚝 떨어지더니 ...
7,11,1,예년보다 따뜻한 11월이더니 역시 겨울은 겨울입니다. 갑자기 기온이 뚝 떨어지더니 ...
8,1,1,예년보다 따뜻한 11월이더니 역시 겨울은 겨울입니다. 갑자기 기온이 뚝 떨어지더니 ...


In [76]:
group_df = sentence_df.groupby(['RawText', 'Aspect_pred']).mean()

In [77]:
group_df

Pola_pred
RawText                                            Aspect_pred           
130년에 가까운 전통 글로벌 스포츠 브랜드인 OO의 트랙수트입니다.  OO이라고 믿... 0                  1.0
                                                   1                  0.0
                                                   8                 -0.5
                                                   10                 1.0
                                                   13                 1.0
...                                                                   ...
히든밴딩과 매끈한 절개라인으로 슬림핏을 연출해 주는 바디라인 레깅스를 소개할려해요. ... 1                  1.0
                                                   5                  1.0
                                                   13                 1.0
                                                   16                 1.0
                                                   17                 1.0

[8362 rows x 1 columns]

In [78]:
group_df.reset_index(inplace=True)

In [79]:
group_df['Aspect_pred'] = le.inverse_transform(group_df['Aspect_pred'])

In [80]:
# group_df['RawText'].unique()

In [81]:
# total_df['RawText'].unique()

In [82]:
# total_df 와 group_df를 join 결합
review_df = pd.merge(total_df, group_df, on = 'RawText', how = 'inner')

In [83]:
# ProductName의 빈도 수 체크
len(review_df['ProductName'].unique())

517

In [84]:
# 제품별 리뷰의 상세 감정 분석이 가능
# 제품 이름 중 가장 많은 리뷰를 가진 제품을 선택하여 감정 점수의 평균을 확인

In [85]:
review_df.head()

,Index,RawText,Source,Domain,MainCategory,ProductName,Syllable,Word,GeneralPolarity,Aspects,Aspect_pred,Pola_pred
0,1037679,안녕하세요. 벌써 7월이라니 시간 정말 빨리 가는거 같아요. 날씨는 또 왜이리 더운...,SNS,패션,여성의류,OO 썸머 루즈핏팬츠,302,63,1.0,"[{'Aspect': '기능', 'SentimentText': '시원하면서', 'S...",기능,1.0
1,1037679,안녕하세요. 벌써 7월이라니 시간 정말 빨리 가는거 같아요. 날씨는 또 왜이리 더운...,SNS,패션,여성의류,OO 썸머 루즈핏팬츠,302,63,1.0,"[{'Aspect': '기능', 'SentimentText': '시원하면서', 'S...",소재,1.0
2,1037679,안녕하세요. 벌써 7월이라니 시간 정말 빨리 가는거 같아요. 날씨는 또 왜이리 더운...,SNS,패션,여성의류,OO 썸머 루즈핏팬츠,302,63,1.0,"[{'Aspect': '기능', 'SentimentText': '시원하면서', 'S...",제품구성,1.0
3,1037679,안녕하세요. 벌써 7월이라니 시간 정말 빨리 가는거 같아요. 날씨는 또 왜이리 더운...,SNS,패션,여성의류,OO 썸머 루즈핏팬츠,302,63,1.0,"[{'Aspect': '기능', 'SentimentText': '시원하면서', 'S...",착용감,1.0
4,1037679,안녕하세요. 벌써 7월이라니 시간 정말 빨리 가는거 같아요. 날씨는 또 왜이리 더운...,SNS,패션,여성의류,OO 썸머 루즈핏팬츠,302,63,1.0,"[{'Aspect': '기능', 'SentimentText': '시원하면서', 'S...",핏,1.0


In [86]:
review_df.groupby(['ProductName'])

In [87]:
pd.pivot_table(
    data = review_df.drop_duplicates('RawText'),
    index = 'ProductName',
    values = 'RawText',
    aggfunc=('count')
).sort_values('RawText', ascending=False).index[0]

'OO 코튼 가디건 '

In [88]:
product_name = 'OO 코튼 가디건 '
# case1 -> ProductName에서 필터링을 한 뒤 Aspect_pred를 기준으로 그룹화 -> Pola_pred의 평균
test_df = review_df.loc[review_df['ProductName'] == product_name, ]
test_df.groupby('Aspect_pred')['Pola_pred'].mean()

Aspect_pred
가격      0.875000
기능      0.944444
길이      1.000000
두께      0.875000
디자인     1.000000
마감      1.000000
사이즈     0.300000
색상      0.950000
소재      1.000000
신축성     1.000000
제품구성   -1.000000
착용감     1.000000
촉감      1.000000
품질      0.600000
핏       1.000000
활용성     0.666667
Name: Pola_pred, dtype: float64

In [89]:
# case2 -> review_df 에서 ProductName과 Aspect_pred를 기준으로 그룹화
# Pola_pred의 평균을 구한다
# 원하는 제품명을 선텍하여 확인
group_df2 = review_df.groupby(['ProductName', 'Aspect_pred'])['Pola_pred'].mean()

In [90]:
group_df2[product_name]

Aspect_pred
가격      0.875000
기능      0.944444
길이      1.000000
두께      0.875000
디자인     1.000000
마감      1.000000
사이즈     0.300000
색상      0.950000
소재      1.000000
신축성     1.000000
제품구성   -1.000000
착용감     1.000000
촉감      1.000000
품질      0.600000
핏       1.000000
활용성     0.666667
Name: Pola_pred, dtype: float64